In [ ]:
import pandas as pd

In [ ]:
def expand_sentences(data):
    """
    Expands the dataset by duplicating or triplicating each sentence based on the availability of up to 
    three sets of SPO labels (spo_label_1, spo_label_2, spo_label_3). Each duplicated sentence is assigned
    a unique identifier to ensure that each instance is treated as distinct during model training.

    """
    # Create a new Df to store the expanded data
    expanded_data = []

    # Iterate over unique sentence_ids
    for sentence_id in data['sentence_id'].unique():
        sentence_data = data[data['sentence_id'] == sentence_id]  # Extract data for the current sentence_id

        # For each set of labels, if the column has non-'_' labels, append the data with these labels
        for index, label in enumerate(['spo_label_1', 'spo_label_2', 'spo_label_3'], start=1):
            if sentence_data[label].str.contains('[^_]').any():
                temp_data = sentence_data.copy() # Create a copy of the sentence data
                temp_data['spo_label'] = temp_data[label] # Assign the relevant labels
                temp_data['sentence_id'] = f"{sentence_id}_{index}" # Assign a new unique sentence ID
                temp_data = temp_data[['sentence_id', 'token_id', 'sentence', 'token', 'spo_label']] # Keep only the necessary columns
                expanded_data.append(temp_data)
    
    # Concatenate all Dfs in the list
    return pd.concat(expanded_data, ignore_index=True)


# Read the datasets
train_data = pd.read_csv('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/final data/final_train_data.csv')
validation_data = pd.read_csv('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/final data/final_validation_data.csv')
test_data = pd.read_csv('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/final data/final_test_data.csv')

# Apply the expansion function
expanded_train_data = expand_sentences(train_data)
expanded_validation_data = expand_sentences(validation_data)
expanded_test_data = expand_sentences(test_data)

accepted_labels = ['Subject', 'Predicate', 'Object', 'other']
expanded_train_data['spo_label'] = expanded_train_data['spo_label'].apply(lambda x: x if x in accepted_labels else 'other')
expanded_validation_data['spo_label'] = expanded_validation_data['spo_label'].apply(lambda x: x if x in accepted_labels else 'other')
expanded_test_data['spo_label'] = expanded_test_data['spo_label'].apply(lambda x: x if x in accepted_labels else 'other')

# Save the new expanded dataset
expanded_train_data.to_csv('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/final data/expanded_train_data.csv', index=False)
expanded_validation_data.to_csv('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/final data/expanded_validation_data.csv', index=False)
expanded_test_data.to_csv('C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/final data/expanded_test_data.csv', index=False)